# NVFP4


Before understanding NVFP4, you first need to understand what actually limits LLM performance.

Most people assume the GPU is busy doing matrix multiplication.

For large language models, that's only partly true.

The real bottleneck is usually:

GPU Memory (HBM)
        ↓
Load weights
        ↓
Tensor Cores
        ↓
Compute
        ↓
Write results

For example:

Hidden size = 8192

Weight matrix:

8192 × 8192

≈67 million weights

If stored in FP16:

67M × 2 bytes
≈134 MB

Why not just use FP4?

The obvious idea is:

FP16 → FP4

2 bytes
↓

0.5 bytes

That gives:

134 MB

↓

33.5 MB

That's a 4× reduction in memory traffic.

Understanding FP4

A floating-point number has three parts:

Sign
Exponent
Mantissa

Conceptually:

value = (-1)^sign × mantissa × 2^exponent

With FP16, you have enough bits to represent a wide range of values.

With FP4, you only have 4 total bits.

A simplified view:

S E E M

1 sign
2 exponent
1 mantissa

Only 16 possible bit patterns exist.

That means only 16 representable values.

For example (illustrative, not exact NVFP4 encoding):

0
±0.5
±1
±2
±4
±8
...

Now consider a real neural-network weight:

0.127

FP4 cannot represent it exactly.

It may become

0.125

or

0.25

depending on the encoding.

That error is called quantization error.

Where NVFP4 fits in

NVFP4 is NVIDIA's hardware-native FP4 format designed specifically for Blackwell Tensor Cores. It combines:

A 4-bit floating-point representation for weights.
Efficient per-block scaling handled directly in hardware.
Tensor Core instructions that can read FP4 values, apply the block scale, and perform matrix multiplication in a single pipeline, accumulating results in higher precision (such as FP16 or FP32).

The crucial point is that the Tensor Cores understand the format natively. Earlier GPUs could emulate low-bit formats in software, but Blackwell performs the unpacking, scaling, multiplication, and accumulation as dedicated hardware operations, dramatically reducing overhead.

What problem is NVFP4 actually solving?

You already know:

FP4 uses only 4 bits.
FP4 saves memory and bandwidth.
FP4 has poor precision.

The real question is:

Why is FP4 inaccurate for neural networks, and how does NVFP4 fix that?

Everything else is built on this.

The real problem is NOT FP4

The problem is using one numeric representation for values that have completely different magnitudes.

Take a weight matrix:

0.12   0.09   0.11
0.10   0.08   0.13
0.09   0.11   0.10

These numbers are all around 0.1.

Now imagine another matrix:

15.2   16.1   14.8
17.0   15.9   16.3
14.7   15.5   16.0

These numbers are around 16.

Can the same FP4 encoding efficiently represent both?

No.

One of them is going to lose precision.

This has nothing to do with AI.

It's simply a numerical representation problem.

Scaling is just normalization

Suppose we have

0.12
0.09
0.11

Choose a scale

Scale = 0.1

Now normalize.

0.12 / 0.1 = 1.2

0.09 / 0.1 = 0.9

0.11 / 0.1 = 1.1

Instead of storing

0.12

store

1.2

Later,

Real Weight

=

Stored Value × Scale
1.2 × 0.1 = 0.12

Nothing magical happened.

Scaling is literally just

Store

weight / scale

Recover

stored × scale

That's it.

Step 4: Granularity of Scaling
Per-Tensor

One scale for the entire tensor.

Tensor

+----------------+
|                |
|                |
|                |
+----------------+

Scale = S

Advantages

Minimal metadata
Fast

Disadvantages

Poor accuracy
One outlier affects the entire tensor
Per-Channel

Each output channel gets its own scale.

For a linear layer

Weight

Rows

S1
S2
S3
S4

Every row has its own scale.

Much better accuracy.

Still relatively cheap.

Widely used in INT8 inference.

Per-Block

Now divide every row into blocks.

Example

□□□□□□□□□□□□□□□□

↓

□□□□ □□□□ □□□□ □□□□

Each block gets

Scale 1

Scale 2

Scale 3

Scale 4

Now every small region adapts independently.

This is block scaling.

Why Block Scaling Works

Inside a small region, weights usually have similar magnitudes.

Example

0.10
0.11
0.12
0.09

One scale fits perfectly.

Compare with

0.10
12.4
0.09
9.8

One scale cannot represent both accurately.

By shrinking the block size, you reduce the variation inside each block.

That reduces quantization error.

Tradeoff

Smaller blocks

↓

More scales

↓

More metadata

↓

Better accuracy

Larger blocks

↓

Fewer scales

↓

Less metadata

↓

More quantization error

Every quantization format chooses a block size based on this tradeoff.

# What Makes NVFP4 Different?
Many people think

NVFP4 = FP4

Wrong.

NVFP4 is roughly

FP4 Values
+
Block Scale
+
Hardware Support

Those three pieces together define the format.

The scale is part of the representation.

What Happens During Matrix Multiplication?

Suppose one block stores

Scale = 0.1

FP4

1.2
0.9
1.1

When Blackwell loads them, it does not first expand the whole matrix into FP16 in memory.

Instead, inside the Tensor Core pipeline it conceptually performs

Load FP4

↓

Read block scale

↓

Reconstruct value

↓

Multiply with activation

↓

Accumulate

The reconstruction happens on the fly, inside the Tensor Core pipeline.

That means the GPU still transfers only FP4-sized data from memory.

This is why NVFP4 reduces bandwidth without paying a large reconstruction cost.

The Mental Model I Want You to Keep

Forget "NVFP4 is a 4-bit floating point."

Think of it as:

NVFP4 Weight

=
{
    FP4 value,
    Block scale,
    Hardware decode rules
}

The FP4 value is almost the least interesting part. The real innovation is that the Tensor Core treats a block of FP4 values and their shared scale as a single computational unit, allowing 4-bit weights to be used with minimal accuracy loss and minimal runtime overhead.

# MXFP4

What does the "MX" mean?
MX = Microscaling

The key idea is:

Instead of one scale for a large block, use much smaller, more local scales.

FP4
    ↓
FP4 + one scale
    ↓
Block Scaling
    ↓
Microscaling (MXFP4)

Microscaling

Instead of
□□□□□□□□□□□□□□□□

↓

One Scale

MXFP4 does

□□□□ □□□□ □□□□ □□□□

S1    S2    S3    S4

But doesn't this increase metadata?

Yes.

## Why is MXFP4 hardware-specific?
If you implemented microscaling in software, every matrix multiplication would look like

Read scale

↓

Apply scale

↓

Read next scale

↓

Apply scale

↓

Repeat...

Lots of instructions.

Lots of branching.

Lots of overhead.

Blackwell Tensor Cores understand the microscale layout directly.

Conceptually,

Load FP4

↓

Load associated microscale

↓

Multiply by scale

↓

Tensor Core FMA

↓

Accumulate

The scaling is fused into the Tensor Core pipeline.

This is why NVIDIA could afford to make scales much more local.

# HARDWARE work

Software implementation

Suppose your weights are

FP4 FP4 FP4 FP4

Your CUDA kernel has to do something like

```c++
uint8 packed = load();

fp4 a = unpack_first(packed);
fp4 b = unpack_second(packed);

float wa = a * scale;
float wb = b * scale;

mma(wa, wb, ...)```

Notice all the extra work:

unpack bits
extract nibbles
convert FP4 → FP16
multiply by scale
only then perform matrix multiply

These are actual instructions executed by CUDA cores or preprocessing logic.

Blackwell Tensor Core receives

FP4
Scale


Software

Imagine driving.

You stop the car.

Open the trunk.

Take out groceries.

Close trunk.

Drive again.

Every step costs time.

Hardware

Imagine a conveyor belt.

As the box moves,

someone automatically opens it,

takes out the groceries,

puts them on another belt,

all while it keeps moving.

No stopping.

The work still happens.

But it isn't an extra step anymore.

## what if u dont want to encode decode? 

Case 1: You send FP32 weights
HBM
 ↓
FP32
 ↓
Tensor Core / CUDA Core
 ↓
FP32 multiply
 ↓
FP32 accumulate

No quantization.

No decode.

No scale.

The Tensor Core simply treats the input as FP32.

Case 2: You send NVFP4 weights
HBM
 ↓
FP4 + Scale
 ↓
Tensor Core
 ↓
Hardware dequantization
 ↓
Multiply
 ↓
Accumulate

Now the Tensor Core knows:

"These operands are NVFP4."

So it automatically performs

unpack FP4
apply scale
reconstruct the value

before multiplying.

The important point is that you don't write the decode kernel. You simply issue an mma.nvfp4-type instruction (or use a library like cuBLAS/cuDNN that does), and the Tensor Core performs the decode internally.